# N03 — ML (robusto) con bloques por modelo

Este notebook permite ejecutar **cada modelo en un bloque independiente**:
- RandomForest
- XGBoost
- LightGBM

Los datasets se eligen automáticamente por horizonte:
- 1m/3m → `df_st_daily_short_plazo.xlsx`
- 6m/12m → `df_st_daily_long_plazo.xlsx`

Cada bloque guarda:
- Resultados por horizonte (CSV/XLSX)
- Predicciones OOS completas por fecha (CSV en `OOS_DIR`)

Al final hay una celda opcional para consolidar y obtener el **mejor modelo por horizonte**.


In [ ]:
# =========================
# 0) Imports + Config
# =========================

import os
import time
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.base import clone

# SVM / SVR
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from scipy.stats import loguniform

# Gradient boosting
try:
    import lightgbm as lgb
    HAS_LGBM = True
except Exception:
    HAS_LGBM = False
    print("Warning: lightgbm no disponible; se salta LGBM.")

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception:
    HAS_XGB = False
    print("Warning: xgboost no disponible; se salta XGB.")

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATE_COL   = "Fecha"
TARGET_COL = "usd_dlog"

HORIZONS = {"1m": 21, "3m": 63, "6m": 126, "12m": 252}

MAX_LAG = 30

# =========================
# SMOKE TEST (rápido)
# =========================
# Si querés saltarlo, poné RUN_SMOKE_TEST = False.
RUN_SMOKE_TEST = True

# Horizontes a probar (subconjunto de HORIZONS). Ajustá si querés.
SMOKE_TAGS = ["1m", "6m"]

# Recorte de filas para que sea rápido (usa tail()).
SMOKE_MAX_ROWS = 600

# Tuning / CV reducido
SMOKE_N_SPLITS_TSCV = 3
SMOKE_N_ITER_SEARCH = 8

# Walk-forward reducido
SMOKE_STEP_OOS = 5

# Logging
SMOKE_PRINT_EVERY = 200

# Para el smoke test, por defecto desactivamos SHAP (rápido y evita incompatibilidades).
SMOKE_DISABLE_SHAP = True

# Columnas esperadas del OOS en NIVEL (cuando hay df_level disponible)
SMOKE_REQUIRED_OOS_COLS_LEVEL = ["Fecha", "Fecha_objetivo", "dolar_t", "dolar_pred_tH", "dolar_real_tH"]


# Smoke test de exports (lectura de archivos): dejar en False salvo debugging
RUN_EXPORT_SMOKE_TEST = True
# =========================
# Directorios de salida
# =========================
BASE_SAVE_DIR = "results_n04"

# Nota: a diferencia de versiones anteriores, el SMOKE TEST NO redirige a otra carpeta.
# Todo se guarda en la misma estructura que el run completo (como en N04 DL).
SAVE_DIR = BASE_SAVE_DIR

OOS_DIR        = os.path.join(SAVE_DIR, "oos")
FI_DIR         = os.path.join(SAVE_DIR, "feature_importance")
PLOTS_DIR      = os.path.join(SAVE_DIR, "plots_levels")
ARTIFACTS_DIR  = os.path.join(SAVE_DIR, "artifacts_ml")

for d in [SAVE_DIR, OOS_DIR, FI_DIR, PLOTS_DIR, ARTIFACTS_DIR]:
    os.makedirs(d, exist_ok=True)

# Carpeta estándar a nivel proyecto (para que N07 cargue sin rutas internas)
PROD_ARTIFACTS_DIR = Path("artifacts_ml")
PROD_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print("OK config.")
print("SAVE_DIR:", SAVE_DIR)
print("OOS_DIR:", OOS_DIR)
print("FI_DIR:", FI_DIR)
print("PLOTS_DIR:", PLOTS_DIR)
print("ARTIFACTS_DIR:", ARTIFACTS_DIR)
print("PROD_ARTIFACTS_DIR:", PROD_ARTIFACTS_DIR)

# =========================
# Lags por frecuencia (diario vs mensual)
# =========================
# Si USE_FREQ_AWARE_LAGS=True, build_xy_by_h usará:
# - daily_lags para variables de frecuencia diaria
# - monthly_lags para variables de frecuencia mensual (aprox 21 días hábiles por mes)
USE_FREQ_AWARE_LAGS = True

# Legacy (solo usado si NO se usa la grilla de lags)
LAG_DAILY  = 2
LAG_MONTHLY = 21

# =========================
# Lags como hiperparámetro (grilla chica por horizonte)
# =========================
def make_lag_grid(tag: str):
    """Devuelve una lista de configuraciones de lags a evaluar para un horizonte (tag)."""
    def r(a, b):
        return list(range(a, b + 1))

    if tag in ("1m", "3m"):
        return [
            {"name": "d1_5_m3_t1_10",  "daily_lags": r(1, 5),  "monthly_lags": [21, 42, 63],                 "target_lags": r(1, 10)},
            {"name": "d1_10_m3_t1_10", "daily_lags": r(1, 10), "monthly_lags": [21, 42, 63],                 "target_lags": r(1, 10)},
            {"name": "d1_10_m6_t1_10", "daily_lags": r(1, 10), "monthly_lags": [21*k for k in range(1, 7)],  "target_lags": r(1, 10)},
        ]

    # 6m / 12m
    return [
        {"name": "d1_10_m6_t1_10",          "daily_lags": r(1, 10), "monthly_lags": [21*k for k in range(1, 7)],   "target_lags": r(1, 10)},
        {"name": "d1_30_m6_t1_20",          "daily_lags": r(1, 30), "monthly_lags": [21*k for k in range(1, 7)],   "target_lags": r(1, 20)},
        {"name": "d1_30_m12_sparse_t1_20",  "daily_lags": r(1, 30), "monthly_lags": [21, 42, 63, 126, 189, 252],   "target_lags": r(1, 20)},
        {"name": "d1_30_m12_full_t1_20",    "daily_lags": r(1, 30), "monthly_lags": [21*k for k in range(1, 13)],  "target_lags": r(1, 20)},
    ]

# Control manual opcional de variables mensuales (si querés forzar una lista)
MONTHLY_FEATURES_OVERRIDE = None
# Heurística: una variable se considera 'mensual' si, por mes calendario, tiene pocos valores distintos
MONTHLY_NUNIQUE_MEDIAN_THRESHOLD = 2.0

INITIAL_TRAIN_FRAC = 0.7

# Tuning
N_SPLITS_TSCV  = 5
N_ITER_SEARCH  = 20

# =========================
# SHAP config
# =========================
DO_SHAP = True
SHAP_N_SAMPLES = 500
SHAP_BG_SAMPLES = 100
SHAP_TOP_K_RAW = 50
SHAP_TOP_K_GROUP = 30


In [ ]:
import os
import json
import joblib
from pathlib import Path

def _maybe_extract_scaler(model):
    """Devuelve un objeto scaler si el modelo es un Pipeline con step 'scaler'. Si no, None."""
    try:
        from sklearn.pipeline import Pipeline
        if isinstance(model, Pipeline) and "scaler" in model.named_steps:
            return model.named_steps["scaler"]
    except Exception:
        pass
    return None

def save_ml_artifacts(model, features, params, h_tag, model_safe, base_save_dir, metadata_extra=None):
    """Guarda artefactos ML con la misma lógica de DL:
    - Doble copia: dentro de results/ (trazabilidad) y en carpeta estándar artifacts_ml/ (producción)
    - Archivos por (h_tag, model): model.joblib, scaler.joblib, features.json, params.json, metadata.json
    """
    base_save_dir = str(base_save_dir)
    art_dir = Path(base_save_dir) / "artifacts_ml" / h_tag / model_safe
    prod_dir = Path("artifacts_ml") / h_tag / model_safe
    art_dir.mkdir(parents=True, exist_ok=True)
    prod_dir.mkdir(parents=True, exist_ok=True)

    # 1) modelo completo (incluye scaler si es Pipeline)
    joblib.dump(model, art_dir / "model.joblib")
    joblib.dump(model, prod_dir / "model.joblib")

    # 2) scaler explícito (para simetría con DL). Si no existe, se guarda None.
    scaler = _maybe_extract_scaler(model)
    joblib.dump(scaler, art_dir / "scaler.joblib")
    joblib.dump(scaler, prod_dir / "scaler.joblib")

    # 3) features (orden exacto)
    (art_dir / "features.json").write_text(json.dumps(list(features), ensure_ascii=False, indent=2))
    (prod_dir / "features.json").write_text(json.dumps(list(features), ensure_ascii=False, indent=2))

    # 4) params
    (art_dir / "params.json").write_text(json.dumps(params, ensure_ascii=False, indent=2, default=str))
    (prod_dir / "params.json").write_text(json.dumps(params, ensure_ascii=False, indent=2, default=str))

    # 5) metadata
    meta = {
        "h_tag": h_tag,
        "H_days": int(params.get("H_days")) if isinstance(params, dict) and params.get("H_days") is not None else None,
        "family": "ML",
        "model": params.get("model_label") if isinstance(params, dict) else None,
        "model_safe": model_safe,
        "target_col": params.get("target_col") if isinstance(params, dict) else globals().get("TARGET_COL"),
        "date_col": globals().get("DATE_COL"),
        "n_features": int(len(features)),
    }
    if metadata_extra:
        meta.update(metadata_extra)

    (art_dir / "metadata.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2, default=str))
    (prod_dir / "metadata.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2, default=str))

    print(f"✅ Artefactos ML guardados en: {art_dir}  (y copia prod: {prod_dir})")



In [ ]:
# =========================
# 1) Carga de datos (short vs long por horizonte)
# =========================

BASE_DIR = Path.cwd()

INPUT_SHORT = BASE_DIR / "results_n02" / "df_st_daily_short_plazo.xlsx"
INPUT_LONG  = BASE_DIR / "results_n02" / "df_st_daily_long_plazo.xlsx"
INPUT_SELECTED = BASE_DIR / "results_n02" / "df_st_daily_general.xlsx"
USE_SINGLE_BASE = False

files_to_check = [INPUT_SHORT, INPUT_LONG]
if USE_SINGLE_BASE:
    files_to_check = [INPUT_SELECTED]

for p in files_to_check:
    if not os.path.isfile(p):
        raise FileNotFoundError(
            f"No se encontró el archivo '{p}'. Subilo/ubicálo y revisá el nombre."
        )

df_short = pd.read_excel(INPUT_SHORT)
df_long  = pd.read_excel(INPUT_LONG)
df_selected = pd.read_excel(INPUT_SELECTED) if USE_SINGLE_BASE else None

def _prepare_df(df: pd.DataFrame) -> pd.DataFrame:
    if DATE_COL not in df.columns:
        raise ValueError(f"No se encontró la columna '{DATE_COL}'")
    if TARGET_COL not in df.columns:
        raise ValueError(f"No se encontró la columna target '{TARGET_COL}'")
    df = df.copy()
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
    df = df.dropna(subset=[DATE_COL]).sort_values(DATE_COL).reset_index(drop=True)
    return df

df_short = _prepare_df(df_short)
df_long  = _prepare_df(df_long)
if USE_SINGLE_BASE:
    df_selected = _prepare_df(df_selected)

print("Short shape:", df_short.shape, "| fechas:", df_short[DATE_COL].min(), "->", df_short[DATE_COL].max())
print("Long  shape:", df_long.shape,  "| fechas:", df_long[DATE_COL].min(),  "->", df_long[DATE_COL].max())

if USE_SINGLE_BASE:
    print("Selected shape:", df_selected.shape, "| fechas:", df_selected[DATE_COL].min(), "->", df_selected[DATE_COL].max())

def get_df_for_tag(tag: str) -> pd.DataFrame:
    if USE_SINGLE_BASE:
        return df_selected

    # 1m/3m => short; 6m/12m => long
    if tag in ("1m", "3m"):
        return df_short
    if tag in ("6m", "12m"):
        return df_long
    raise ValueError(f"Horizonte desconocido: {tag}")



In [ ]:
# =========================
# 1B) Serie en NIVEL del tipo de cambio (para des-transformar OOS a nivel)
# =========================
# Usamos la misma lógica del N06: cargar un dataframe con ['Fecha','dolar'].
# Si el archivo no existe, el notebook sigue corriendo (solo no calcula métricas/plots en nivel).

BASE_DIR = Path.cwd()

# Ruta esperada (ajustá si tu N00 vive en otro lado)
DF_LEVEL_FILE = os.path.join(BASE_DIR, "results_n00", "df_daily.xlsx")

df_level = None
if os.path.isfile(DF_LEVEL_FILE):
    df_level = pd.read_excel(DF_LEVEL_FILE, usecols=["Fecha", "dolar"]).copy()
    df_level["Fecha"] = pd.to_datetime(df_level["Fecha"])
    df_level = df_level.sort_values("Fecha").drop_duplicates("Fecha").reset_index(drop=True)
    print("✅ Cargado nivel desde:", DF_LEVEL_FILE)
else:
    print("⚠️ No se encontró df_daily.xlsx con nivel (dolar). Se omiten métricas y gráficos en NIVEL.")
    print("   Busqué en:", DF_LEVEL_FILE)

def add_level_targets_from_oos(oos_df: pd.DataFrame, df_level: pd.DataFrame, H_days: int) -> pd.DataFrame:
    """Une nivel en Fecha, crea Fecha_objetivo (t+H), observado y predicho en nivel.

    Retorna columnas estándar:
      - Fecha (t)
      - Fecha_objetivo (t+H)
      - dolar_t
      - dolar_pred_tH
      - dolar_real_tH
      - y_true (log-return acumulado)
      - y_pred (log-return acumulado)
    """
    tmp = oos_df.copy()
    tmp["Fecha"] = pd.to_datetime(tmp["Fecha"])

    mapH = df_level.copy()
    mapH["Fecha_objetivo"] = mapH["Fecha"].shift(-int(H_days))
    mapH["dolar_real_tH"] = mapH["dolar"].shift(-int(H_days))
    mapH = mapH.rename(columns={"dolar": "dolar_t"})
    mapH = mapH[["Fecha", "dolar_t", "Fecha_objetivo", "dolar_real_tH"]]

    out = tmp.merge(mapH, on="Fecha", how="left")

    # Inversión a nivel (usd_dlog es log-diff; el target y_* es suma futura a H)
    out["dolar_pred_tH"] = out["dolar_t"] * np.exp(out["y_pred"])

    out = out.dropna(subset=["dolar_real_tH", "dolar_pred_tH", "Fecha_objetivo", "dolar_t"]).reset_index(drop=True)
    return out

def level_metrics(level_df: pd.DataFrame) -> dict:
    y_obs = level_df["dolar_real_tH"].astype(float)
    y_hat = level_df["dolar_pred_tH"].astype(float)
    err = y_obs - y_hat
    return {
        "MAE_level": float(np.mean(np.abs(err))),
        "RMSE_level": float(np.sqrt(np.mean(err**2))),
        "R2_level": float(r2_score(y_obs, y_hat)),
        "n_level": int(len(level_df))
    }

def plot_level_series(level_df: pd.DataFrame, model_name: str, h_tag: str, save_dir: str):
    """Grafica real vs predicho en NIVEL usando Fecha_objetivo como eje."""
    import matplotlib.pyplot as plt

    dfp = level_df.sort_values("Fecha_objetivo").copy()
    fig = plt.figure(figsize=(12, 3.8))
    plt.plot(dfp["Fecha_objetivo"], dfp["dolar_real_tH"], label="Real (nivel)")
    plt.plot(dfp["Fecha_objetivo"], dfp["dolar_pred_tH"], label=f"Predicho (nivel) - {model_name}")
    plt.title(f"{h_tag} | Comparación en nivel (dólar)")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()

    safe = model_name.replace("(", "").replace(")", "").replace(" ", "")
    out_png_dir = os.path.join(save_dir, safe)
    os.makedirs(out_png_dir, exist_ok=True)
    fpath = os.path.join(out_png_dir, f"level_{h_tag}_{safe}.png")
    plt.savefig(fpath, dpi=160)
    plt.show()
    return fpath


In [ ]:
# =========================
# 2) Funciones
# =========================

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def future_sum(series: pd.Series, H: int) -> pd.Series:
    """r_H(t) = sum_{j=1..H} r_{t+j}, alineado en t."""
    base = series.astype(float)
    return base.shift(-1).rolling(H).sum().shift(-(H - 1))


def infer_monthly_features(df: pd.DataFrame, feature_cols, threshold_median_nunique: float = 1.0):
    """Heurística para detectar variables 'mensuales' en un dataset diario.

    Si dentro de cada mes calendario la variable cambia poco (mediana de nunique <= threshold),
    la tratamos como mensual (en este pipeline: se usa un lag de 21 días).

    Nota: es una heurística robusta y *sin* metadata; podés fijar manualmente MONTHLY_FEATURES_OVERRIDE
    en la config si querés control total.
    """
    if len(feature_cols) == 0:
        return set()

    m = pd.to_datetime(df[DATE_COL]).dt.to_period("M")
    monthly = set()
    for c in feature_cols:
        try:
            nun = df.groupby(m)[c].nunique(dropna=True)
            if float(nun.median()) <= float(threshold_median_nunique):
                monthly.add(c)
        except Exception:
            # si falla por tipo raro, no lo marcamos mensual
            pass
    return monthly



def build_xy_by_h(
    df: pd.DataFrame,
    target_col: str,
    H: int,
    max_lag: int = 30,
    use_freq_aware_lags: bool = True,
    lag_daily: int = 2,
    lag_monthly: int = 21,
    daily_lags=None,
    monthly_lags=None,
    target_lags=None,
    monthly_features_override=None,
    monthly_threshold_median_nunique: float = 1.0
):
    """Construye X, y y fechas para un horizonte H (misma lógica que N03).

    Soporta dos modos:

    1) use_freq_aware_lags=True (recomendado):
       - Variables mensuales: usa lags en `monthly_lags` (por defecto [lag_monthly])
       - Variables diarias:   usa lags en `daily_lags`   (por defecto [lag_daily])
       - Target (rezagos del propio target): usa `target_lags` (por defecto = daily_lags)

       Nota: este modo permite predecir todos los días, usando el último dato disponible
       para variables mensuales (p.ej. 21 días) y, además, incorporar "memoria" con pocos
       rezagos (p.ej. 21/42/63 para mensuales).

    2) use_freq_aware_lags=False (legacy):
       - Comportamiento original: crea todos los lags 1..max_lag para TODOS los predictores.
    """
    df = df.copy()
    y = future_sum(df[target_col], H).rename(f"r_{H}")

    feats = [c for c in df.columns
             if c not in [target_col, DATE_COL] and pd.api.types.is_numeric_dtype(df[c])]

    X_parts = []

    if use_freq_aware_lags:
        # Defaults (mantienen compatibilidad con el comportamiento previo freq-aware)
        if daily_lags is None:
            daily_lags = [lag_daily]
        if monthly_lags is None:
            monthly_lags = [lag_monthly]
        if target_lags is None:
            target_lags = list(daily_lags)

        # normalizar/ordenar para evitar duplicados
        daily_lags = sorted(set(int(x) for x in daily_lags))
        monthly_lags = sorted(set(int(x) for x in monthly_lags))
        target_lags = sorted(set(int(x) for x in target_lags))

        if monthly_features_override is not None:
            monthly_feats = set(monthly_features_override)
        else:
            monthly_feats = infer_monthly_features(
                df, feats,
                threshold_median_nunique=monthly_threshold_median_nunique
            )

        # Lags para predictores
        for c in feats:
            if not pd.api.types.is_numeric_dtype(df[c]):
                continue
            lag_list = monthly_lags if c in monthly_feats else daily_lags
            for L in lag_list:
                X_parts.append(df[c].shift(L).rename(f"{c}_L{L}"))

        # Lags del propio target (siempre se tratan como "diarios", salvo que elijas otra lista)
        for L in target_lags:
            X_parts.append(df[target_col].shift(L).rename(f"{target_col}_L{L}"))

    else:
        # Comportamiento original: todos los lags 1..max_lag para todos los predictores
        for c in feats + [target_col]:
            if pd.api.types.is_numeric_dtype(df[c]):
                for L in range(1, max_lag + 1):
                    X_parts.append(df[c].shift(L).rename(f"{c}_L{L}"))

    X = pd.concat(X_parts, axis=1)
    XY = pd.concat([df[DATE_COL], X, y], axis=1).dropna()

    dates_clean = XY[DATE_COL].reset_index(drop=True)
    y_clean = XY[y.name].reset_index(drop=True)
    X_clean = XY.drop(columns=[DATE_COL, y.name]).reset_index(drop=True)
    return X_clean, y_clean, dates_clean



def time_series_random_search(
    X, y, estimator, param_dist,
    n_splits,
    n_iter,
    scoring="neg_mean_absolute_error",
    random_state=42,
    verbose=2
):
    """RandomizedSearchCV respetando orden temporal (TimeSeriesSplit)."""
    tss = TimeSeriesSplit(n_splits=n_splits)
    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=param_dist,
        n_iter=n_iter,
        cv=tss,
        scoring=scoring,
        random_state=random_state,
        n_jobs=-1,
        verbose=verbose
    )
    search.fit(X, y)
    return search.best_estimator_, search.best_params_, search.best_score_

def expanding_oos_predictions_with_dates(
    X, y, dates,
    base_estimator,
    initial_train_size: int,
    step: int = 1,
    every: int = 50,
    model_name: str = "",
    tag: str = ""
):
    """OOS con ventana expansiva (misma lógica que N03)."""
    y_true_all, y_pred_all, dates_all = [], [], []
    total = max(1, (len(y) - initial_train_size + step - 1) // step)
    t0 = time.perf_counter()

    for i, end in enumerate(range(initial_train_size, len(y), step), 1):
        est = clone(base_estimator)
        est.fit(X.iloc[:end], y.iloc[:end])
        y_hat = est.predict(X.iloc[[end]])[0]

        y_true_all.append(float(y.iloc[end]))
        y_pred_all.append(float(y_hat))
        dates_all.append(dates.iloc[end])

        if (i % every == 0) or (i == total):
            elapsed = time.perf_counter() - t0
            prefix = f"[{model_name}@{tag}] " if (model_name or tag) else ""
            print(f"{prefix}{i}/{total} steps | {elapsed:,.1f}s", flush=True)

    return np.array(y_true_all), np.array(y_pred_all), np.array(dates_all)


In [ ]:
# =========================
# 3) Modelos + espacios de búsqueda (definidos ANTES de usarse)
# =========================

def rf_model_and_space(random_state=RANDOM_STATE):
    model = RandomForestRegressor(
        random_state=random_state,
        n_estimators=500,
        n_jobs=-1
    )
    space = {
        "n_estimators": [300, 500, 800],
        "max_depth": [None, 6, 10, 16],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2"],
    }
    return model, space


def xgb_model_and_space(random_state=RANDOM_STATE):
    if not HAS_XGB:
        return None, None
    model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=800,
        random_state=random_state,
        tree_method="hist",
        n_jobs=-1
    )
    space = {
        "max_depth": [3, 4, 6, 8],
        "learning_rate": [0.01, 0.03, 0.05],
        "subsample": [0.6, 0.8, 1.0],
        "colsample_bytree": [0.6, 0.8, 1.0],
        "min_child_weight": [1, 3, 5, 10],
        "reg_alpha": [0.0, 0.1, 0.5],
        "reg_lambda": [0.0, 0.1, 0.5],
    }
    return model, space


def lgbm_model_and_space(random_state=RANDOM_STATE):
    if not HAS_LGBM:
        return None, None
    model = lgb.LGBMRegressor(
        boosting_type="gbdt",
        objective="regression",
        n_estimators=800,
        random_state=random_state,
        n_jobs=-1
    )
    space = {
        "num_leaves": [31, 63, 127],
        "learning_rate": [0.01, 0.03, 0.05],
        "max_depth": [-1, 6, 10, 14],
        "subsample": [0.7, 0.85, 1.0],
        "colsample_bytree": [0.6, 0.8, 1.0],
        "min_data_in_leaf": [10, 30, 60],
        "reg_alpha": [0.0, 0.1, 0.5],
        "reg_lambda": [0.0, 0.1, 0.5]
    }
    return model, space



# -------------------------
# SVR (Support Vector Regression)
# -------------------------

def svr_rbf_model_and_space(random_state=RANDOM_STATE):
    """SVR (RBF) con escalado — versión estándar para forecasting supervisado con lags."""
    model = Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("svr", SVR(kernel="rbf"))
    ])
    space = {
        "svr__C": loguniform(1e-2, 1e3),
        "svr__epsilon": loguniform(1e-4, 1e-1),
        "svr__gamma": loguniform(1e-4, 1e-1),
    }
    return model, space


In [ ]:
# =========================
# 4) SHAP global importance (igual estilo N03)
# =========================

def _base_feature_name(feat: str) -> str:
    if "_L" in feat:
        return feat.split("_L")[0]
    return feat

def shap_global_importance(
    fitted_model,
    X_ref: pd.DataFrame,
    h_tag: str,
    H_days: int,
    model_name: str,
    fi_dir: str,
    shap_n_samples: int = 500,
    bg_n_samples: int = 100,
    top_k_raw: int = 50,
    top_k_group: int = 30,
    random_state: int = 42
):
    """Calcula SHAP (TreeExplainer) y guarda top importances.

    - Si shap no está instalado o el modelo no es compatible, se salta sin romper.
    - Para evitar costos, usa un subconjunto de filas recientes.
    """
    try:
        import shap
    except Exception as e:
        print(f"⚠️ shap no disponible: {e}. Saltando SHAP para {model_name}@{h_tag}.")
        return

    n = len(X_ref)
    if n <= 20:
        print(f"⚠️ Muy pocas filas para SHAP en {model_name}@{h_tag}.")
        return

    n_shap = min(int(shap_n_samples), n)
    X_shap = X_ref.iloc[-n_shap:].copy()
    n_bg = min(int(bg_n_samples), len(X_shap))
    X_bg = X_shap.iloc[:n_bg].copy()

    try:
        explainer = shap.TreeExplainer(fitted_model, data=X_bg)
        shap_vals = explainer.shap_values(X_shap)
    except Exception as e:
        print(f"⚠️ SHAP no soportado para {model_name}@{h_tag}: {e}.")
        return

    if isinstance(shap_vals, list):
        shap_vals = shap_vals[0]
    shap_vals = np.asarray(shap_vals)

    if shap_vals.ndim != 2 or shap_vals.shape[1] != X_shap.shape[1]:
        print(f"⚠️ SHAP shape raro en {model_name}@{h_tag}: {shap_vals.shape} vs X {X_shap.shape}.")
        return

    imp = np.mean(np.abs(shap_vals), axis=0)

    df_raw = pd.DataFrame({
        "h_tag": h_tag,
        "H_days": H_days,
        "model": model_name,
        "feature": X_shap.columns,
        "shap_mean_abs": imp
    }).sort_values("shap_mean_abs", ascending=False).reset_index(drop=True)

    df_raw["rank"] = np.arange(1, len(df_raw) + 1)
    safe_name = model_name.replace("(", "").replace(")", "").replace(" ", "")

    raw_path = os.path.join(fi_dir, f"shap_importance_raw_{h_tag}_{safe_name}.csv")
    df_raw.head(top_k_raw).to_csv(raw_path, index=False)

    df_raw["feature_base"] = df_raw["feature"].map(_base_feature_name)
    df_grp = (
        df_raw.groupby(["h_tag", "H_days", "model", "feature_base"], as_index=False)["shap_mean_abs"]
            .sum()
            .sort_values("shap_mean_abs", ascending=False)
            .reset_index(drop=True)
    )
    df_grp["rank"] = np.arange(1, len(df_grp) + 1)
    grp_path = os.path.join(fi_dir, f"shap_importance_grouped_{h_tag}_{safe_name}.csv")
    df_grp.head(top_k_group).to_csv(grp_path, index=False)

    print(f"✅ SHAP guardado: {raw_path}")
    print(f"✅ SHAP (grouped) guardado: {grp_path}")


In [ ]:
# =========================
# 5) Runner genérico por (modelo, horizonte)
# =========================

STEP_OOS = 1
PRINT_EVERY = 50

def run_model_block(model_label: str, base_model_fn, space_fn):
    """Ejecuta TODO el flujo N03 para un único modelo (para TODOS los horizontes).

    Salidas por horizonte:
      - OOS en OOS_DIR: oos_{h_tag}_{model}.csv y .xlsx
        * Si df_level está disponible => incluye columnas en NIVEL:
          Fecha, Fecha_objetivo, dolar_t, dolar_pred_tH, dolar_real_tH (+ y_true/y_pred)
        * Si NO => guarda (Fecha, y_true, y_pred) (log-return acumulado)
      - Plot en NIVEL (si hay df_level): PLOTS_DIR/{model}/level_{h_tag}_{model}.png
      - SHAP (si DO_SHAP): FI_DIR/shap_importance_*.csv
      - Artefactos ML (modelo final fit en TODO el dataset del horizonte):
        artifacts_ml/{h_tag}/{model}/...
    """

    results_rows = []
    oos_store = {}  # (tag, model) -> (y_true, y_pred, dates)

    t0_block = time.time()
    print("\n" + "="*60)
    print(f"BLOQUE MODELO: {model_label}")
    print("="*60, flush=True)

    for tag, H in HORIZONS.items():

        df = get_df_for_tag(tag)

        print("\n------------------------------", flush=True)
        print(f"Horizonte: {tag} (H={H} días) | dataset={'short' if tag in ('1m','3m') else 'long'}", flush=True)
        print("------------------------------", flush=True)

        # =========================
        # Selección de LAGS
        # =========================
        lag_grid = make_lag_grid(tag) if USE_FREQ_AWARE_LAGS else [
            {"name": f"legacy_1_{MAX_LAG}", "daily_lags": None, "monthly_lags": None, "target_lags": None}
        ]

        best_overall = None

        for lag_cfg in lag_grid:

            X, y, dates = build_xy_by_h(
                df, TARGET_COL, H,
                max_lag=MAX_LAG,
                use_freq_aware_lags=USE_FREQ_AWARE_LAGS,
                lag_daily=LAG_DAILY,
                lag_monthly=LAG_MONTHLY,
                daily_lags=lag_cfg.get("daily_lags"),
                monthly_lags=lag_cfg.get("monthly_lags"),
                target_lags=lag_cfg.get("target_lags"),
                monthly_features_override=MONTHLY_FEATURES_OVERRIDE,
                monthly_threshold_median_nunique=MONTHLY_NUNIQUE_MEDIAN_THRESHOLD
            )

            if len(y) < 200:
                continue

            n0 = int(len(y) * INITIAL_TRAIN_FRAC)
            X_tr, y_tr = X.iloc[:n0], y.iloc[:n0]

            base_model = base_model_fn()
            space = space_fn()

            if base_model is None or space is None:
                print(f"⚠️ {model_label} no disponible. Salteo.")
                best_overall = None
                break

            print(f"🔎 Search START | {model_label} | {tag} | lag_cfg={lag_cfg['name']}", flush=True)
            t_s0 = time.time()

            best_est, best_params, best_score = time_series_random_search(
                X_tr, y_tr,
                estimator=base_model,
                param_dist=space,
                n_splits=N_SPLITS_TSCV,
                n_iter=N_ITER_SEARCH,
                scoring="neg_mean_absolute_error",
                random_state=RANDOM_STATE,
                verbose=2
            )

            t_s1 = time.time()
            search_min = (t_s1 - t_s0) / 60.0

            print(
                f"✅ Search END | {model_label} | {tag} | lag_cfg={lag_cfg['name']} | "
                f"best_cv_score={best_score:.6f} | min={search_min:.2f}",
                flush=True
            )

            if (best_overall is None) or (best_score > best_overall["best_score"]):
                best_overall = dict(
                    lag_cfg=lag_cfg,
                    best_est=best_est,
                    best_params=best_params,
                    best_score=best_score,
                    X=X, y=y, dates=dates,
                    n0=n0, X_tr=X_tr, y_tr=y_tr,
                    search_min=search_min
                )

        if best_overall is None:
            print("⚠️ No se pudo seleccionar configuración de lags. Salteo.", flush=True)
            continue

        # Recuperar mejor configuración
        lag_cfg     = best_overall["lag_cfg"]
        best_est    = best_overall["best_est"]      # ya viene fit en X_tr por RandomizedSearchCV
        best_params = best_overall["best_params"]
        best_score  = best_overall["best_score"]
        X           = best_overall["X"]
        y           = best_overall["y"]
        dates       = best_overall["dates"]
        n0          = best_overall["n0"]
        X_tr        = best_overall["X_tr"]
        y_tr        = best_overall["y_tr"]
        search_min  = best_overall["search_min"]

        print(f"🏁 BEST LAGS | {model_label} | {tag} => {lag_cfg['name']} | cv_score={best_score:.6f}", flush=True)

        # =========================
        # Walk-forward
        # =========================
        expected_steps = (len(y) - n0 + STEP_OOS - 1) // STEP_OOS
        print(f"🚶 Walk-forward START | {model_label} | {tag} | steps≈{expected_steps}", flush=True)

        t_w0 = time.time()
        y_true, y_pred, dts = expanding_oos_predictions_with_dates(
            X, y, dates,
            base_estimator=best_est,
            initial_train_size=n0,
            step=STEP_OOS,
            every=PRINT_EVERY,
            model_name=model_label,
            tag=tag
        )
        t_w1 = time.time()

        print(f"✅ Walk-forward END | {model_label} | {tag} | min={(t_w1 - t_w0)/60.0:.2f}", flush=True)

        # =========================
        # Guardar OOS (log y/o nivel)
        # =========================
        safe_name = model_label.replace("(", "").replace(")", "").replace(" ", "")

        oos_log_df = pd.DataFrame({"Fecha": dts, "y_true": y_true, "y_pred": y_pred})
        oos_csv  = os.path.join(OOS_DIR, f"oos_{tag}_{safe_name}.csv")
        oos_xlsx = os.path.join(OOS_DIR, f"oos_{tag}_{safe_name}.xlsx")

        if df_level is not None:
            try:
                lvl_df = add_level_targets_from_oos(oos_log_df, df_level, H_days=H)
                # Guardar con columnas estándar + log-returns (útiles para debug)
                out_cols = ["Fecha", "Fecha_objetivo", "dolar_t", "dolar_pred_tH", "dolar_real_tH", "y_true", "y_pred"]
                lvl_df[out_cols].to_csv(oos_csv, index=False)
                lvl_df[out_cols].to_excel(oos_xlsx, index=False)

                # Plot
                try:
                    plot_level_series(lvl_df, model_name=model_label, h_tag=tag, save_dir=PLOTS_DIR)
                except Exception as e:
                    print(f"⚠️ No pude generar plot NIVEL {model_label}@{tag}: {e}")

            except Exception as e:
                print(f"⚠️ Error al construir/guardar OOS en NIVEL {model_label}@{tag}: {e}")
                # fallback log
                oos_log_df.to_csv(oos_csv, index=False)
                oos_log_df.to_excel(oos_xlsx, index=False)
        else:
            oos_log_df.to_csv(oos_csv, index=False)
            oos_log_df.to_excel(oos_xlsx, index=False)

        # =========================
        # Result row (solo NIVEL; si no hay df_level queda NaN)
        # =========================
        r = {
            "h_tag": tag,
            "H_days": H,
            "model": model_label,
            "MAE_level": np.nan,
            "RMSE_level": np.nan,
            "R2_level": np.nan,
            "params": str(best_params),
            "best_cv_score": float(best_score),
            "lag_cfg": lag_cfg.get("name"),
            "daily_lags": str(lag_cfg.get("daily_lags")),
            "monthly_lags": str(lag_cfg.get("monthly_lags")),
            "target_lags": str(lag_cfg.get("target_lags")),
            "search_min": float(search_min),
            "oos_min": float((t_w1 - t_w0)/60.0),
            "n_oos": int(len(y_true))
        }

        if df_level is not None:
            try:
                m = level_metrics(pd.read_csv(oos_csv))
                r["MAE_level"] = m["MAE_level"]
                r["RMSE_level"] = m["RMSE_level"]
                r["R2_level"] = m["R2_level"]
            except Exception as e:
                print(f"⚠️ Error cálculo métricas NIVEL {model_label}@{tag}: {e}")

        results_rows.append(r)
        oos_store[(tag, model_label)] = (y_true, y_pred, dts)

        # =========================
        # SHAP (si aplica)
        # =========================
        if DO_SHAP:
            try:
                shap_global_importance(
                    fitted_model=best_est,
                    X_ref=X_tr,
                    h_tag=tag,
                    H_days=H,
                    model_name=model_label,
                    fi_dir=FI_DIR,
                    shap_n_samples=SHAP_N_SAMPLES,
                    bg_n_samples=SHAP_BG_SAMPLES,
                    top_k_raw=SHAP_TOP_K_RAW,
                    top_k_group=SHAP_TOP_K_GROUP,
                    random_state=RANDOM_STATE
                )
            except Exception as e:
                print(f"⚠️ SHAP falló para {model_label}@{tag}: {e}")

        # =========================
        # Guardar ARTEFACTOS para producción (fit en TODO el dataset del horizonte)
        # =========================
        try:
            final_model = clone(best_est)
            final_model.fit(X, y)
            save_ml_artifacts(
                model=final_model,
                features=list(X.columns),
                params={
                    "best_params": best_params,
                    "best_cv_score": float(best_score),
                    "model_label": model_label,
                    "lag_cfg": lag_cfg,
                    "target_col": TARGET_COL,
                    "H_days": int(H),
                    "h_tag": tag
                },
                h_tag=tag,
                model_safe=safe_name,
                base_save_dir=SAVE_DIR,
                metadata_extra={
                    "trained_until": str(pd.to_datetime(dates).max()),
                    "n_rows_xy": int(len(y)),
                    "initial_train_size": int(n0),
                    "step_oos": int(STEP_OOS),
                }
            )
        except Exception as e:
            print(f"⚠️ No pude guardar artefactos {model_label}@{tag}: {e}")

        # =========================
        # Guardar resultados parciales
        # =========================
        results_df = pd.DataFrame(results_rows).sort_values(["h_tag"]).reset_index(drop=True)

        out_csv  = os.path.join(SAVE_DIR, f"fx_results_{safe_name}.csv")
        out_xlsx = os.path.join(SAVE_DIR, f"fx_results_{safe_name}.xlsx")

        results_df.to_csv(out_csv, index=False)
        results_df.to_excel(out_xlsx, index=False)

    t1_block = time.time()

    print("\n✅ Bloque", model_label, "terminado. Total (min):", (t1_block - t0_block)/60.0)
    print("Resultados guardados en:", out_csv)

    return results_df, oos_store


In [ ]:
# (Configuración del SMOKE TEST movida a la celda 0)


In [ ]:
# =========================
# 5B) Smoke test (recomendado antes de correr horas)
# =========================
# Objetivo: asegurar que (tuning -> walk-forward -> guardados -> NIVEL -> plots -> artefactos) funcione
# para cada modelo, sin esperar horas para descubrir un error.

import os
import pandas as pd
from pathlib import Path

def _safe_model_name(model_label: str) -> str:
    return model_label.replace("(", "").replace(")", "").replace(" ", "")

def run_smoke_tests():
    global N_SPLITS_TSCV, N_ITER_SEARCH, STEP_OOS, PRINT_EVERY, DO_SHAP, HORIZONS
    global get_df_for_tag

    print("\n" + "="*60)
    print("SMOKE TEST: START")
    print("="*60)

    # 1) Guardar y luego restaurar globals (solo tuning + horizons + getter)
    _old = {
        "N_SPLITS_TSCV": N_SPLITS_TSCV,
        "N_ITER_SEARCH": N_ITER_SEARCH,
        "STEP_OOS": STEP_OOS,
        "PRINT_EVERY": PRINT_EVERY,
        "DO_SHAP": DO_SHAP,
        "HORIZONS": dict(HORIZONS),
        "get_df_for_tag": get_df_for_tag,
    }

    # override tuning / walk-forward
    N_SPLITS_TSCV = SMOKE_N_SPLITS_TSCV
    N_ITER_SEARCH = SMOKE_N_ITER_SEARCH
    STEP_OOS      = SMOKE_STEP_OOS
    PRINT_EVERY   = SMOKE_PRINT_EVERY
    if SMOKE_DISABLE_SHAP:
        DO_SHAP = False

    # horizons subset
    HORIZONS = {k: _old["HORIZONS"][k] for k in SMOKE_TAGS if k in _old["HORIZONS"]}
    if len(HORIZONS) == 0:
        raise ValueError("SMOKE_TAGS no coincide con HORIZONS.")

    # modelos a testear (mismos wrappers que los bloques)
    model_specs = []

    def _rf_base():
        m, _ = rf_model_and_space()
        return m
    def _rf_space():
        _, s = rf_model_and_space()
        return s
    model_specs.append(("RandomForest", _rf_base, _rf_space))

    if HAS_XGB:
        def _xgb_base():
            m, _ = xgb_model_and_space()
            return m
        def _xgb_space():
            _, s = xgb_model_and_space()
            return s
        model_specs.append(("XGBoost", _xgb_base, _xgb_space))

    if HAS_LGBM:
        def _lgb_base():
            m, _ = lgbm_model_and_space()
            return m
        def _lgb_space():
            _, s = lgbm_model_and_space()
            return s
        model_specs.append(("LightGBM", _lgb_base, _lgb_space))

    def _svr_base():
        m, _ = svr_rbf_model_and_space()
        return m
    def _svr_space():
        _, s = svr_rbf_model_and_space()
        return s
    model_specs.append(("SVR_RBF", _svr_base, _svr_space))

    # 2) Ejecutar smoke por modelo
    for model_label, base_fn, space_fn in model_specs:
        print("\n" + "-"*60)
        print(f"SMOKE | {model_label}")
        print("-"*60)

        # recortar datos dentro del getter (sin tocar tu lógica general)
        _old_get = _old["get_df_for_tag"]

        def get_df_for_tag_smoke(tag: str):
            df0 = _old_get(tag)
            if df0 is None:
                return None
            return df0.tail(SMOKE_MAX_ROWS).reset_index(drop=True)

        # monkeypatch temporal
        globals()["get_df_for_tag"] = get_df_for_tag_smoke
        try:
            _res_df, _ = run_model_block(model_label, base_fn, space_fn)
        finally:
            globals()["get_df_for_tag"] = _old_get

        # 3) Validaciones mínimas de artefactos por horizonte
        safe = _safe_model_name(model_label)

        fx_csv = os.path.join(SAVE_DIR, f"fx_results_{safe}.csv")
        assert os.path.isfile(fx_csv), f"Falta archivo resultados: {fx_csv}"

        for tag in HORIZONS.keys():
            oos_csv = os.path.join(OOS_DIR, f"oos_{tag}_{safe}.csv")
            assert os.path.isfile(oos_csv), f"Falta OOS CSV: {oos_csv}"
            df_oos = pd.read_csv(oos_csv)

            # Si hay df_level, exigimos columnas NIVEL; si no, exigimos columnas log
            if df_level is not None:
                missing = [c for c in SMOKE_REQUIRED_OOS_COLS_LEVEL if c not in df_oos.columns]
                assert len(missing) == 0, f"OOS {oos_csv} no tiene columnas NIVEL requeridas. Faltan: {missing}"
            else:
                missing = [c for c in ["Fecha", "y_true", "y_pred"] if c not in df_oos.columns]
                assert len(missing) == 0, f"OOS {oos_csv} no tiene columnas log requeridas. Faltan: {missing}"

            # plot de nivel (opcional)
            plot_png = os.path.join(PLOTS_DIR, safe, f"level_{tag}_{safe}.png")
            if df_level is not None and not os.path.isfile(plot_png):
                print(f"⚠️ (SMOKE) No encontré plot: {plot_png} (opcional)")

            # artefactos
            prod_dir = Path("artifacts_ml") / tag / safe / "model.joblib"
            if not prod_dir.is_file():
                print(f"⚠️ (SMOKE) No encontré artefacto prod: {prod_dir} (opcional)")

        print(f"✅ SMOKE OK | {model_label}")

    # 4) Restore globals
    N_SPLITS_TSCV = _old["N_SPLITS_TSCV"]
    N_ITER_SEARCH = _old["N_ITER_SEARCH"]
    STEP_OOS      = _old["STEP_OOS"]
    PRINT_EVERY   = _old["PRINT_EVERY"]
    DO_SHAP       = _old["DO_SHAP"]
    HORIZONS      = _old["HORIZONS"]
    globals()["get_df_for_tag"] = _old["get_df_for_tag"]

    print("\n" + "="*60)
    print("SMOKE TEST: END (todo OK)")
    print("="*60)
    print("Carpeta de outputs:", SAVE_DIR)

# Ejecutar smoke test si está habilitado (recomendado)
if RUN_SMOKE_TEST:
    run_smoke_tests()


In [ ]:
# =========================
# 6A) BLOQUE: RandomForest (ejecutar esta celda para correr SOLO RF)
# =========================

def _rf_base():
    m, _ = rf_model_and_space()
    return m

def _rf_space():
    _, s = rf_model_and_space()
    return s

rf_results_df, rf_oos_store = run_model_block("RandomForest", _rf_base, _rf_space)
display(rf_results_df)


In [ ]:
# =========================
# 6B) BLOQUE: XGBoost (ejecutar esta celda para correr SOLO XGB)
# =========================

if HAS_XGB:
    def _xgb_base():
        m, _ = xgb_model_and_space()
        return m

    def _xgb_space():
        _, s = xgb_model_and_space()
        return s

    xgb_results_df, xgb_oos_store = run_model_block("XGBoost", _xgb_base, _xgb_space)
    display(xgb_results_df)
else:
    print("⚠️ XGBoost no disponible (HAS_XGB=False)")


In [ ]:
# =========================
# 6C) BLOQUE: LightGBM (ejecutar esta celda para correr SOLO LGBM)
# =========================

if HAS_LGBM:
    def _lgbm_base():
        m, _ = lgbm_model_and_space()
        return m

    def _lgbm_space():
        _, s = lgbm_model_and_space()
        return s

    lgbm_results_df, lgbm_oos_store = run_model_block("LightGBM", _lgbm_base, _lgbm_space)
    display(lgbm_results_df)
else:
    print("⚠️ LightGBM no disponible (HAS_LGBM=False)")


In [ ]:
# =========================
# 6E) BLOQUE: SVR_RBF (ejecutar esta celda para correr SOLO SVR_RBF)
# =========================

def _svr_base():
    m, _ = svr_rbf_model_and_space()
    return m

def _svr_space():
    _, s = svr_rbf_model_and_space()
    return s

svr_results_df, svr_oos_store = run_model_block("SVR_RBF", _svr_base, _svr_space)
display(svr_results_df)


In [ ]:
# =========================
# 7) Consolidar (opcional): mejor modelo por horizonte
# =========================
# Esta celda NO entrena nada. Lee los resultados ya guardados por bloque (RF/XGB/LGBM)
# y construye:
# - fx_results_ml.csv/xlsx (consolidado)
# - fx_best_model_by_horizon.csv/xlsx
# - OOS del mejor por horizonte en OOS_DIR/best_by_horizon/

result_files = []
for name in ["RandomForest", "XGBoost", "LightGBM", "SVR_RBF"]:
    safe = name.replace("(", "").replace(")", "").replace(" ", "")
    f = os.path.join(SAVE_DIR, f"fx_results_{safe}.csv")
    if os.path.isfile(f):
        result_files.append(f)

if len(result_files) == 0:
    raise FileNotFoundError("No encuentro resultados por modelo. Corré al menos un bloque (RF/XGB/LGBM).")

all_results = pd.concat([pd.read_csv(f) for f in result_files], ignore_index=True)
all_results = all_results.sort_values(["h_tag", "MAE_level", "model"]).reset_index(drop=True)

out_all_csv  = os.path.join(SAVE_DIR, "fx_results_ml.csv")
out_all_xlsx = os.path.join(SAVE_DIR, "fx_results_ml.xlsx")
all_results.to_csv(out_all_csv, index=False)
all_results.to_excel(out_all_xlsx, index=False)

best_idx = all_results.groupby("h_tag")["MAE_level"].idxmin()
best_by_h = all_results.loc[best_idx].sort_values("H_days").reset_index(drop=True)

best_csv  = os.path.join(SAVE_DIR, "fx_best_model_by_horizon.csv")
best_xlsx = os.path.join(SAVE_DIR, "fx_best_model_by_horizon.xlsx")
best_by_h.to_csv(best_csv, index=False)
best_by_h.to_excel(best_xlsx, index=False)

print("✅ Consolidado guardado:", out_all_csv)
print("✅ Best-by-horizon guardado:", best_csv)
display(best_by_h)

best_oos_dir = os.path.join(OOS_DIR, "best_by_horizon")
os.makedirs(best_oos_dir, exist_ok=True)

for _, r in best_by_h.iterrows():
    tag = r["h_tag"]
    model_name = r["model"]
    safe = model_name.replace("(", "").replace(")", "").replace(" ", "")
    src_oos = os.path.join(OOS_DIR, f"oos_{tag}_{safe}.csv")
    if not os.path.isfile(src_oos):
        print("⚠️ Falta OOS:", src_oos)
        continue
    df_o = pd.read_csv(src_oos)
    df_o.to_csv(os.path.join(best_oos_dir, f"oos_best_{tag}.csv"), index=False)
    # Copiar también XLSX si existe
    src_oos_xlsx = os.path.join(OOS_DIR, f"oos_{tag}_{safe}.xlsx")
    if os.path.isfile(src_oos_xlsx):
        try:
            pd.read_excel(src_oos_xlsx).to_excel(os.path.join(best_oos_dir, f"oos_best_{tag}.xlsx"), index=False)
        except Exception as e:
            print("⚠️ No pude copiar XLSX:", src_oos_xlsx, e)

print("✅ OOS best-by-horizon en:", best_oos_dir)


## Guardado de Artefactos para Producción
Esta sección permite guardar el modelo entrenado junto con las features utilizadas y sus parámetros finales. No modifica el entrenamiento existente.

In [ ]:
# (Funciones de guardado de artefactos movidas arriba, antes del runner)


In [ ]:
# =========================
# 8) Smoke test de EXPORTS (lee todo lo exportado)
# =========================
# Este smoke test NO entrena: valida que los archivos exportados existan y sean legibles.
# Incluye: fx_results_ml, OOS (csv+xlsx), plots, y artefactos (results/ y prod/).

import os
from pathlib import Path
import pandas as pd
import joblib

def _assert_file(p: str):
    if not os.path.isfile(p):
        raise FileNotFoundError(p)

def smoke_test_exports():
    base = Path(SAVE_DIR)
    oos_dir = Path(OOS_DIR)
    plots_dir = Path(PLOTS_DIR)

    # 1) Consolidado
    fx_csv  = base / "fx_results_ml.csv"
    fx_xlsx = base / "fx_results_ml.xlsx"
    _assert_file(str(fx_csv)); _assert_file(str(fx_xlsx))

    fx_df = pd.read_csv(fx_csv)
    fx_df_x = pd.read_excel(fx_xlsx)

    if len(fx_df) == 0:
        raise ValueError("fx_results_ml.csv está vacío.")
    if len(fx_df_x) == 0:
        raise ValueError("fx_results_ml.xlsx está vacío.")

    # 2) Por modelo (si existen)
    per_model = list(base.glob("fx_results_*.csv"))
    if len(per_model) == 0:
        print("⚠️ No encuentro fx_results_*.csv por modelo (no es fatal).")

    # 3) Verificar OOS + plots + artefactos por fila de fx_results_ml
    missing = {"oos_csv": [], "oos_xlsx": [], "plot": [], "artifacts_results": [], "artifacts_prod": []}

    for _, r in fx_df.iterrows():
        tag = r["h_tag"]
        model = r["model"]
        safe = str(model).replace("(", "").replace(")", "").replace(" ", "")

        # OOS
        oos_csv = oos_dir / f"oos_{tag}_{safe}.csv"
        oos_xlsx = oos_dir / f"oos_{tag}_{safe}.xlsx"
        if not oos_csv.is_file():  missing["oos_csv"].append(str(oos_csv))
        if not oos_xlsx.is_file(): missing["oos_xlsx"].append(str(oos_xlsx))

        # Plot (si tu función usa otro naming, ajustá acá)
        plot_png = plots_dir / safe / f"level_{tag}_{safe}.png"
        if not plot_png.is_file():
            # tolerante: buscar cualquier png que contenga tag y safe
            cand = list((plots_dir / safe).glob(f"*{tag}*{safe}*.png"))
            if len(cand) == 0:
                missing["plot"].append(str(plot_png))

        # Artefactos (results + prod)
        art_res = Path(SAVE_DIR) / "artifacts_ml" / tag / safe
        art_prod = Path("artifacts_ml") / tag / safe

        needed = ["model.joblib", "scaler.joblib", "features.json", "params.json", "metadata.json"]
        if not all((art_res / f).is_file() for f in needed):
            missing["artifacts_results"].append(str(art_res))
        if not all((art_prod / f).is_file() for f in needed):
            missing["artifacts_prod"].append(str(art_prod))

    # 4) Lectura rápida de un OOS + un modelo (sanity)
    sample = fx_df.iloc[0]
    tag = sample["h_tag"]; model = sample["model"]
    safe = str(model).replace("(", "").replace(")", "").replace(" ", "")
    oos_csv = oos_dir / f"oos_{tag}_{safe}.csv"
    if oos_csv.is_file():
        dfo = pd.read_csv(oos_csv)
        if len(dfo) == 0:
            raise ValueError(f"OOS vacío: {oos_csv}")
        required_cols = {"Fecha", "Fecha_objetivo", "dolar_t", "dolar_pred_tH", "dolar_real_tH"}
        if not required_cols.issubset(set(dfo.columns)):
            print("⚠️ OOS columnas no estándar (no fatal). Faltan:", required_cols - set(dfo.columns))

    model_path = Path("artifacts_ml") / tag / safe / "model.joblib"
    if model_path.is_file():
        _ = joblib.load(model_path)  # solo carga para verificar que no está corrupto

    # Reporte final
    n_miss = sum(len(v) for v in missing.values())
    if n_miss > 0:
        print("⚠️ Smoke test encontró faltantes:")
        for k, v in missing.items():
            if len(v) > 0:
                print(f" - {k}: {len(v)} (ej: {v[0]})")
        raise RuntimeError("Smoke test FAILED: faltan exports/artefactos.")
    else:
        print("✅ Smoke test OK: exports completos y legibles.")
        print(" -", fx_csv)
        print(" -", fx_xlsx)
        print(" - OOS:", oos_dir)
        print(" - Plots:", plots_dir)
        print(" - Artifacts prod:", Path('artifacts_ml'))

# Ejecutar (solo si querés validar exports; por defecto apagado)
if "RUN_EXPORT_SMOKE_TEST" in globals() and RUN_EXPORT_SMOKE_TEST:
    smoke_test_exports()
